# Final results and paper figures (Colab, no GPU needed)

**On the Jetson:** `python3 pack_results.py` → upload the `results_pack_<stamp>.zip` here, then Runtime → Run all.

Figures (saved in `figs/`, all 300 dpi PNG + PDF) and tables (`tables/`):
1. Part A: positioning error as planned → verified (all coverage runs) + Wilcoxon test
2. SOGA vs current strategies: model coverage over part orientations (no-motion analysis)
3. SOGA vs current strategies: **measured** glare-free coverage, path and time on the robot
4. Coverage maps and unrolled surfaces (examples)
5. Autonomous inspection: confusion matrix, accuracy per condition, classifier vs surface model vs combined,
   ROC of the surface anomaly score, time and views per part
6. Defect localisation examples (anomaly heat map + overlay)
7. RL / classifier results from Colab training (if present)
8. One table with every number quoted in the paper

Runs whose tag starts with `dry` are excluded automatically.

In [ ]:
import glob, os, zipfile, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image
if not glob.glob('/content/results_pack_*.zip'):
    from google.colab import files; files.upload()
z = sorted(glob.glob('/content/results_pack_*.zip'))[-1]
os.makedirs('/content/R', exist_ok=True); zipfile.ZipFile(z).extractall('/content/R'); os.chdir('/content/R')
os.makedirs('figs', exist_ok=True); os.makedirs('tables', exist_ok=True)
plt.rcParams.update({'font.size': 9, 'font.family': 'serif', 'axes.grid': True, 'grid.alpha': .3, 'savefig.dpi': 300})
def save(fig, name):
    fig.tight_layout(); fig.savefig('figs/%s.png' % name); fig.savefig('figs/%s.pdf' % name); plt.show()
dry = lambda p: os.path.basename(os.path.dirname(p)).startswith('dry')
NUM = {}
print('unpacked', z)
for k in ['refcover', 'soga', 'autoinspect', 'viewplan']:
    print('  %-12s %d runs' % (k, len([d for d in glob.glob('hw_logs/%s/*/' % k) if not os.path.basename(d[:-1]).startswith('dry')])))

In [ ]:
# 1) Part A: positioning error as planned -> after feed-forward -> verified
PARTA_PREFIX = ('cover', 'ring_cover')      # which coverage runs to use (tags starting with these); () = all
need = {'view_err_raw_mm', 'view_err_open_mm', 'view_err_mm', 'kappa_meas'}
fs = [f for f in sorted(glob.glob('hw_logs/refcover/*/views.csv')) if not dry(f)
      and (not PARTA_PREFIX or os.path.basename(os.path.dirname(f)).startswith(PARTA_PREFIX))
      and need <= set(pd.read_csv(f, nrows=1).columns)]
print('Part A runs used:', [os.path.basename(os.path.dirname(f)) for f in fs])
if fs:
    V = pd.concat([pd.read_csv(f).assign(run=os.path.basename(os.path.dirname(f))) for f in fs], ignore_index=True)
    cols = [('view_err_raw_mm', 'as planned'), ('view_err_open_mm', 'after feed-forward'), ('view_err_mm', 'verified (closed loop)')]
    fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.8))
    data = [V[c].dropna().values for c, _ in cols]
    ax[0].boxplot(data, showfliers=True); ax[0].set_xticks([1, 2, 3]); ax[0].set_xticklabels([l for _, l in cols], fontsize=7); ax[0].set_ylabel('view error (mm)')
    for i, d in enumerate(data): ax[0].scatter(np.random.normal(i + 1, .04, len(d)), d, s=6, alpha=.5)
    ax[0].set_title('Part A: view positioning (n = %d views, %d runs)' % (len(V), V.run.nunique()), fontsize=8)
    for c, l in cols: ax[1].hist(V[c].dropna(), bins=20, alpha=.5, label=l)
    ax[1].set_xlabel('view error (mm)'); ax[1].legend(fontsize=7)
    save(fig, 'fig1_partA_positioning')
    from scipy.stats import wilcoxon
    ok = V[['view_err_raw_mm', 'view_err_mm']].dropna()
    p = wilcoxon(ok.view_err_raw_mm, ok.view_err_mm).pvalue
    NUM.update(partA_views=len(V), partA_runs=V.run.nunique(), partA_planned_median_mm=V.view_err_raw_mm.median(),
               partA_verified_median_mm=V.view_err_mm.median(), partA_verified_p95_mm=V.view_err_mm.quantile(.95),
               partA_wilcoxon_p=p, partA_kappa_min=V.kappa_meas.min())
    print({k: NUM[k] for k in NUM if k.startswith('partA')})
else:
    print('no refcover runs')

In [ ]:
# 2) SOGA (no-motion analysis): glare-free coverage of the upper surface vs part orientation
f = 'results_soga/soga_analyze.json'
if os.path.exists(f):
    A = json.load(open(f)); rows = A['rows']
    meths = [k for k in rows[0] if k not in ('placement', 'beta')]
    lab = [r['placement'].replace('axis ', '') for r in rows]
    fig, ax = plt.subplots(1, 2, figsize=(7.4, 2.9), gridspec_kw={'width_ratios': [2.2, 1]})
    w = .8 / len(meths)
    for j, m in enumerate(meths):
        ax[0].bar(np.arange(len(rows)) + j * w, [100 * r[m]['glarefree_upper'] for r in rows], w, label=m)
    ax[0].set_xticks(np.arange(len(rows)) + .4 - w / 2); ax[0].set_xticklabels(lab, rotation=30, fontsize=7)
    ax[0].set_ylabel('glare-free coverage (%)'); ax[0].legend(fontsize=6, ncol=2); ax[0].set_title('model, by part orientation', fontsize=8)
    for m in meths:
        ax[1].scatter([r[m]['path_mm'] for r in rows], [100 * r[m]['glarefree_upper'] for r in rows], s=14, label=m)
    ax[1].set_xlabel('camera path (mm)'); ax[1].set_ylabel('glare-free coverage (%)'); ax[1].set_title('coverage vs motion', fontsize=8)
    save(fig, 'fig2_soga_model')
    T = pd.DataFrame({m: {'coverage_mean_%': np.mean([100 * r[m]['glarefree_upper'] for r in rows]),
                          'coverage_min_%': np.min([100 * r[m]['glarefree_upper'] for r in rows]),
                          'views_mean': np.mean([r[m]['n'] for r in rows]), 'path_mean_mm': np.mean([r[m]['path_mm'] for r in rows])}
                      for m in meths}).T.round(1)
    display(T); T.to_csv('tables/soga_model.csv')
    NUM.update(soga_model_cov=T.loc['SOGA', 'coverage_mean_%'], soga_model_path=T.loc['SOGA', 'path_mean_mm'])
else:
    print('run  python3 soga.py analyze $OPTS  on the Jetson first')

In [ ]:
# 3) SOGA vs current strategies on the ROBOT (measured on the real images)
fs = [f for f in sorted(glob.glob('hw_logs/soga/*/soga_run.json')) if not dry(f)]
if fs:
    S = pd.DataFrame([dict(run=json.load(open(f))['tag'], method=json.load(open(f))['method'],
                           glarefree=100 * json.load(open(f))['measured']['glarefree_upper'],
                           seen=100 * json.load(open(f))['measured']['seen_upper'], path=json.load(open(f))['path_mm'],
                           time=json.load(open(f))['wall_s'], views=json.load(open(f))['n_views']) for f in fs])
    G = S.drop(columns=['run']).groupby('method').agg(['mean', 'std', 'count']).round(1); display(G); G.to_csv('tables/soga_measured.csv')
    fig, ax = plt.subplots(1, 3, figsize=(7.4, 2.5))
    order = [m for m in ['soga', 'pitch', 'ring', 'plan'] if m in set(S.method)]
    names = {'soga': 'SOGA (ours)', 'pitch': 'pitch +-10', 'ring': 'isotropic', 'plan': 'Part-A plan'}
    for a_, col, yl in zip(ax, ['glarefree', 'path', 'time'], ['glare-free coverage (%)', 'camera path (mm)', 'time (s)']):
        m_ = [S[S.method == m][col].mean() for m in order]; s_ = [S[S.method == m][col].std() for m in order]
        a_.bar(range(len(order)), m_, yerr=s_, capsize=3, color=['C3' if m == 'soga' else 'C7' for m in order])
        a_.set_xticks(range(len(order))); a_.set_xticklabels([names[m] for m in order], rotation=25, fontsize=7); a_.set_ylabel(yl)
    save(fig, 'fig3_soga_measured')
    from scipy.stats import mannwhitneyu
    for m in order:
        if m != 'soga' and (S.method == 'soga').sum() >= 3 and (S.method == m).sum() >= 3:
            p = mannwhitneyu(S[S.method == 'soga'].glarefree, S[S.method == m].glarefree, alternative='greater').pvalue
            print('SOGA > %-6s glare-free coverage: Mann-Whitney p = %.4f' % (m, p)); NUM['soga_vs_%s_p' % m] = p
    NUM.update(soga_measured_cov=S[S.method == 'soga'].glarefree.mean() if (S.method == 'soga').any() else np.nan)
else:
    print('no soga runs yet (python3 soga.py run --method soga|pitch|ring|plan)')

In [ ]:
# 4) Coverage maps / unrolled surfaces: one example per method (rows: phi +90 top .. -90 bottom; columns: along the axis)
ex = {}
for f in sorted(glob.glob('hw_logs/soga/*/soga_run.json')):
    if dry(f): continue
    m = json.load(open(f))['method']; ex.setdefault(m, os.path.dirname(f))
if ex:
    fig, ax = plt.subplots(2, len(ex), figsize=(2.2 * len(ex), 3.6), squeeze=False)
    for j, (m, d) in enumerate(ex.items()):
        for i, nm in enumerate(['unrolled_surface.png', 'coverage_map.png']):
            p = os.path.join(d, nm)
            if os.path.exists(p): ax[i, j].imshow(plt.imread(p)[..., :3][..., ::-1] if p.endswith('jpg') else plt.imread(p)[..., :3])
            ax[i, j].set_xticks([]); ax[i, j].set_yticks([]); ax[i, j].grid(False)
        ax[0, j].set_title(m, fontsize=8)
    ax[0, 0].set_ylabel('unrolled'); ax[1, 0].set_ylabel('coverage\n(red = glare only)')
    save(fig, 'fig4_unrolled_examples')

In [ ]:
# 5) Autonomous inspection: verdict accuracy, confusion, classifier vs surface model, ROC, time
fs = [f for f in sorted(glob.glob('hw_logs/autoinspect/*/run.json')) if not dry(f)]
R = pd.DataFrame([json.load(open(f)) for f in fs]) if fs else pd.DataFrame()
if len(R):
    R = R[~R.learn_good.astype(bool)].copy()
if len(R):
    R['pred'] = (R.verdict == 'REJECT').astype(int); R['true'] = R.truth_defective.astype(int)
    from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
    cm = confusion_matrix(R.true, R.pred, labels=[0, 1])
    fig, ax = plt.subplots(1, 4, figsize=(9.6, 2.6))
    ax[0].imshow(cm, cmap='Blues'); ax[0].grid(False)
    for (i, j), v in np.ndenumerate(cm): ax[0].text(j, i, v, ha='center', va='center', fontsize=10)
    ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(['ACCEPT', 'REJECT']); ax[0].set_yticks([0, 1]); ax[0].set_yticklabels(['good', 'defective'])
    ax[0].set_xlabel('verdict'); ax[0].set_ylabel('truth'); ax[0].set_title('accuracy %.0f%% (n=%d)' % (100 * R.correct.mean(), len(R)), fontsize=8)
    acc = R.groupby('condition').correct.mean() * 100
    ax[1].bar(range(len(acc)), acc.values); ax[1].set_xticks(range(len(acc))); ax[1].set_xticklabels(acc.index, rotation=35, fontsize=6)
    ax[1].set_ylabel('correct (%)'); ax[1].set_ylim(0, 105)
    src = {}
    for k, nm in [('classifier_reject', 'classifier'), ('surface_reject', 'surface model'), ('pred', 'combined')]:
        v = R[R[k].notna()]
        if len(v): src[nm] = 100 * np.mean(v[k].astype(int) == v.true)
    ax[2].bar(range(len(src)), list(src.values()), color=['C0', 'C2', 'C3'][:len(src)]); ax[2].set_xticks(range(len(src)))
    ax[2].set_xticklabels(list(src.keys()), rotation=20, fontsize=7); ax[2].set_ylabel('accuracy (%)'); ax[2].set_ylim(0, 105)
    s = R.anomaly_max_over_tau.astype(float)
    if s.notna().sum() > 3 and R.true.nunique() == 2:
        ok = s.notna(); fpr, tpr, _ = roc_curve(R.true[ok], s[ok]); auc = roc_auc_score(R.true[ok], s[ok])
        ax[3].plot(fpr, tpr, 'C3'); ax[3].plot([0, 1], [0, 1], 'k:', lw=.8); ax[3].set_title('surface anomaly ROC, AUC %.2f' % auc, fontsize=8)
        ax[3].set_xlabel('false positive rate'); ax[3].set_ylabel('true positive rate'); NUM['auto_anomaly_auc'] = auc
    save(fig, 'fig5_autonomous_inspection')
    fig, ax = plt.subplots(1, 3, figsize=(7.4, 2.4))
    ax[0].hist(R.wall_s, bins=12); ax[0].set_xlabel('time per part (s)')
    ax[1].hist(R.n_views, bins=range(0, int(R.n_views.max()) + 2)); ax[1].set_xlabel('views per part (incl. re-looks)')
    ax[2].hist([np.mean(g) * 100 for g in R.glarefree_upper], bins=12); ax[2].set_xlabel('glare-free coverage per pass (%)')
    save(fig, 'fig5b_autonomous_effort')
    tp, tn = int(cm[1, 1]), int(cm[0, 0]); fp, fn = int(cm[0, 1]), int(cm[1, 0])
    NUM.update(auto_n=len(R), auto_accuracy=R.correct.mean(), auto_recall=tp / max(tp + fn, 1), auto_specificity=tn / max(tn + fp, 1),
               auto_time_median_s=R.wall_s.median(), auto_views_median=R.n_views.median(), **{'auto_acc_' + k: v / 100 for k, v in src.items()})
    display(R[['tag', 'condition', 'truth', 'verdict', 'correct', 'n_views', 'wall_s']])
    R.drop(columns=['items'], errors='ignore').to_csv('tables/autoinspect_runs.csv', index=False)
else:
    print('no autonomous test runs yet')

In [ ]:
# 6) Defect localisation examples: survey, unrolled surface, anomaly heat map, overlay (defective runs)
ds = [os.path.dirname(f) for f in sorted(glob.glob('hw_logs/autoinspect/*/run.json')) if not dry(f)
      and json.load(open(f)).get('truth_defective')][:4]
for d in ds:
    p = sorted(glob.glob(os.path.join(d, 'pass_*')))[0]
    ims = [os.path.join(p, n) for n in ['survey.jpg', 'unrolled_surface.png', 'anomaly_heat.png', 'defects_overlay.png']]
    ims = [i for i in ims if os.path.exists(i)]
    fig, ax = plt.subplots(1, len(ims), figsize=(2.6 * len(ims), 2.3))
    for a_, i in zip(np.atleast_1d(ax), ims):
        im = plt.imread(i); a_.imshow(im[..., :3][..., ::-1] if i.endswith('.jpg') else im[..., :3]); a_.set_title(os.path.basename(i), fontsize=7)
        a_.set_xticks([]); a_.set_yticks([]); a_.grid(False)
    R_ = json.load(open(os.path.join(d, 'run.json')))
    fig.suptitle('%s | truth: %s | %s' % (R_['tag'], R_['truth'], R_['verdict']), fontsize=8)
    save(fig, 'fig6_defects_%s' % R_['tag'])

In [ ]:
# 7) RL policy and classifier results from Colab training (task_colab.py), if packed
for f in ['results_task/classifier_cv.json', 'results_task/task_eval.json', 'results_task/task_library.json']:
    if os.path.exists(f):
        print('=====', f); print(json.dumps(json.load(open(f)), indent=1)[:3000])
for f in sorted(glob.glob('results_task/*.png')) + sorted(glob.glob('paper/figs/*task*.png')):
    print(f); display(Image(f, width=600))

In [ ]:
# 8) Every number for the paper in one table, and download
T = pd.Series(NUM, dtype=object); display(T.to_frame('value')); T.to_csv('tables/paper_numbers.csv')
import shutil; shutil.make_archive('/content/paper_figures_final', 'zip', '/content/R', '.')
from google.colab import files; files.download('/content/paper_figures_final.zip')